In [0]:
file = "/Workspace/Users/anidesifitriaaaa@gmail.com/Drafts/bronze_layer_data"
df_silver = spark.read.format("parquet").option("inferSchema", "true").load(file)

display(df_silver.limit(10))

In [0]:
total_rows = df_silver.count()
print(f"Silver layer total rows: {total_rows}")

In [0]:
from pyspark.sql.functions import expr

df_silver = df_silver.withColumn("Name", expr(
     "initcap(Name)"))
display(df_silver)

In [0]:
from pyspark.sql.functions import col, expr, datediff, trim, initcap, regexp_replace

df_silver1 = df_silver.withColumn(
    "Billing Amount", expr("cast(`Billing Amount` as decimal(10,2))")).withColumn(
    "Estimated Length of Stay", datediff(col("Discharge Date"), col("Date of Admission"))).withColumn(
    "Patient ID", expr("uuid()"))

display(df_silver1)

In [0]:
df_silver2 = df_silver1.withColumn(
    "Hospital", regexp_replace(col("Hospital"), "[^a-zA-Z0-9 ]", " ")).withColumn(
    "Hospital", regexp_replace(col("Hospital"), "\\b[Aa]nd\\b", "")).withColumn(
    "Hospital", trim(regexp_replace(col("Hospital"), " +", " "))).withColumn(
    "Hospital", expr("initcap(Hospital)")).withColumn(
    "Insurance Provider", regexp_replace(col("Insurance Provider"), "UnitedHealthcare", "United Healthcare")).drop("Name")

display(df_silver2)

In [0]:
df_silver3 = df_silver2.withColumn("Hospital", regexp_replace(col("Hospital"), "^(Ltd|Llc|Inc|Plc)\\s+(.*)$", "$2 $1"))

display(df_silver3)

In [0]:
df_silver_final = df_silver3.withColumn(
    "Hospital", regexp_replace(col("Hospital"), "(?i)\\b(\\w+)\\s+\\1\\b", "$1"))

display(df_silver_final)

In [0]:
#remove space and replace with underscore for delta format
new_columns = [col_name.replace(" ", "_") for col_name in df_silver2.columns]
df_silver_final = df_silver2.toDF(*new_columns)

silver_path = "/Workspace/Users/anidesifitriaaaa@gmail.com/Drafts/silver_layer_data"

df_silver_final.write.format("delta").mode("overwrite").save(silver_path)

print("Data Silver Layer Written")

In [0]:
display(df_silver_final)